# Drivers Graphs

This notebook is made to fetch driver telemetry data in a single track in a single year

In [ ]:
import fastf1
import logging
logging.getLogger('fastf1').setLevel(logging.ERROR)
session = fastf1.get_session(2024, 'Bahrain', 'R')
session.load()
[r["DriverId"] for _, r in session.results.iterrows()]

req         WARNING 	DEFAULT CACHE ENABLED! (3.6 GB) /home/matheus/.cache/fastf1
core           INFO 	Loading data for Bahrain Grand Prix - Race [v3.8.1]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	No cached data found for weather_data. Loading data...
_api           INFO 	Fetching weather data...
req            INFO 	Data has been written to cache!
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading 

['max_verstappen',
 'perez',
 'sainz',
 'leclerc',
 'russell',
 'norris',
 'hamilton',
 'piastri',
 'alonso',
 'stroll',
 'zhou',
 'kevin_magnussen',
 'ricciardo',
 'tsunoda',
 'albon',
 'hulkenberg',
 'ocon',
 'gasly',
 'bottas',
 'sargeant']

In [ ]:
import fastf1
import os
import logging
logging.getLogger('fastf1').setLevel(logging.ERROR)

YEAR = 2024
DRIVERS = [
    'albon', 'sainz', 'leclerc', 'alonso', 'hulkenberg'
]

schedule = fastf1.get_event_schedule(YEAR)
events = schedule["EventName"].to_list()
events = [event for event in events if "Testing" not in event]
races = [idx+1 for idx, _ in enumerate(events)]

for race in races:
    for driver in DRIVERS:
        session = fastf1.get_session(YEAR, race, "R")
        session.load(laps=True, telemetry=True, weather=False)
        driver_slug = f"{driver}_{YEAR}"
        for _, row in session.results.iterrows():
            if row["DriverId"] == driver:
                driver_abv = row["Abbreviation"]
                break
        driver_laps = session.laps.pick_drivers(driver_abv)
        if driver_laps.empty:
            continue
        fastest_lap = driver_laps.pick_fastest()
        if fastest_lap is None:
            fastest_lap = driver_laps.iloc[0]
        start_td = fastest_lap["Time"] - fastest_lap["LapTime"]
        end_td = fastest_lap["Time"]
        telemetry = fastest_lap.get_telemetry()
        telemetry['RelativeSeconds'] = (telemetry['SessionTime'] - start_td).dt.total_seconds()
        telemetry['Compound'] = fastest_lap['Compound']

        position_points = []
        for row in telemetry.itertuples():
            position_points.append((
                round(row.RelativeSeconds, 4), 
                row.X, 
                row.Y,
                row.Z,
                row.RPM, 
                row.Speed, 
                row.nGear, 
                row.Throttle, 
                row.Brake, 
                row.DRS, 
                row.Status,
                row.Compound
            ))

        os.makedirs('data', exist_ok=True)
        filename = f'data/telemetry_{driver}_{YEAR}_{race}.csv'

        with open(filename, 'w') as f:
            f.write("seconds,x,y,z,rpm,speed,gear,throttle,brake,drs,status,compound\n")
            for point in position_points:
                f.write(','.join(map(str, point)) + '\n')

        print(f"Exported {len(position_points)} points for {driver} in {race}")

Exported 737 points for albon in 1
Exported 729 points for sainz in 1
Exported 702 points for leclerc in 1
Exported 700 points for alonso in 1
Exported 733 points for hulkenberg in 1
Exported 724 points for albon in 2
Exported 724 points for sainz in 2
Exported 716 points for leclerc in 2
Exported 718 points for alonso in 2
Exported 720 points for hulkenberg in 2
Exported 624 points for albon in 3
Exported 616 points for sainz in 3
Exported 598 points for leclerc in 3
Exported 622 points for alonso in 3
Exported 618 points for hulkenberg in 3
Exported 991 points for albon in 4
Exported 709 points for sainz in 4
Exported 731 points for leclerc in 4
Exported 719 points for alonso in 4
Exported 733 points for hulkenberg in 4
Exported 742 points for albon in 5
Exported 745 points for sainz in 5
Exported 765 points for leclerc in 5
Exported 756 points for alonso in 5
Exported 794 points for hulkenberg in 5
Exported 690 points for albon in 6
Exported 689 points for sainz in 6
Exported 696 po

IndexError: single positional indexer is out-of-bounds